# Train the AV contradiction detector on AVHBench (Kaggle GPU)

One notebook for the whole run: clone the code, download AVHBench, build leakage-safe splits,
extract frozen CLIP + CLAP features, train every model variant over 3 seeds, pick the best model
on **validation** macro-F1, and report it once on the held-out **test** split.

**Before you run**
1. *Settings → Accelerator*: **GPU** (T4 or P100). *Settings → Internet*: **On** (pip, GitHub, Google Drive, Hugging Face).
2. `REPO_URL` in the next cell points at the project repo. For a **private** repo add a GitHub token as a Kaggle secret named
   `GITHUB_TOKEN` (*Add-ons → Secrets*); it is never printed or saved.
3. AVHBench states **no license**. Use it for research only and check the terms with its authors
   (https://github.com/kaist-ami/AVHBench).
4. Use *Save Version → Save & Run All* so `/kaggle/working/output` is kept as the notebook output.

Labels come from AVHBench's **AV Matching** task: `Yes` → consistent, `No` → contradictory. This is
audio-visual *correspondence*, the closest public proxy for semantic contradiction.

In [ ]:
REPO_URL = "https://github.com/adhvaith267/Cross-Attention-Tensor-Fusion-for-Detecting-Audio-Visual-Semantic-Contradictions-in-Short-Form-Video.git"
BRANCH = "main"
SEEDS = "1 2 3"
GROUP_REGEX = ""   # set in step 4 if swapped clips share an id prefix with their originals

OUT = "/kaggle/working/output"   # kept after the session: manifest, features, checkpoints, reports
CODE = "/tmp/av_contradiction"   # code and raw videos live outside /kaggle/working (not saved)
DATA = "/tmp/avhbench"

## 1. Get the code

In [ ]:
import os
import subprocess

url, token = REPO_URL, None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # no secret attached: public repo
subprocess.run(["rm", "-rf", CODE], check=True)
r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CODE], capture_output=True, text=True)
log = r.stdout + r.stderr
print(log.replace(token, "***") if token else log)
assert r.returncode == 0, "Clone failed: check REPO_URL, BRANCH and the GITHUB_TOKEN secret"
subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", REPO_URL], check=True)  # keep the token off disk
os.chdir(CODE)

## 2. Install and check the GPU

In [ ]:
!pip install -q -e ".[encoders,logging]" gdown
!python -m av_contradiction.cli.check_environment

## 3. Download AVHBench

Downloads the videos and QA annotations from the Google Drive links in the AVHBench README. If
Google Drive refuses (quota), download them yourself, upload them as a private Kaggle Dataset whose
name contains `avhbench`, attach it with *Add Input*, and rerun this cell: it then uses
`/kaggle/input/*avhbench*` instead.

In [ ]:
import glob
import json
import os
import shutil

import gdown

VIDEOS_ID, QA_ID = "10-Qp8zxA3ITT-ileEnCgJkf5Nzx1wry7", "1KcYDAv9lLy3hsx5rWdfRqMFV2NYcZ94W"
attached = glob.glob("/kaggle/input/*avhbench*")
os.makedirs(DATA, exist_ok=True)
if attached:
    src_root = attached[0]
else:
    for fid in (VIDEOS_ID, QA_ID):
        try:
            paths = [gdown.download(id=fid, output=DATA + "/", quiet=False)]
        except Exception:
            paths = gdown.download_folder(id=fid, output=DATA, quiet=False) or []
        for p in paths:
            if p and p.endswith((".zip", ".tar", ".tar.gz", ".tgz")):
                shutil.unpack_archive(p, DATA)
                os.remove(p)
    src_root = DATA

# Collect every clip into one flat folder and every AV-matching question into one json.
VIDS, QA = f"{DATA}/all_videos", f"{DATA}/av_matching.json"
os.makedirs(VIDS, exist_ok=True)
for p in glob.glob(f"{src_root}/**/*.mp4", recursive=True):
    link = os.path.join(VIDS, os.path.basename(p))
    if not os.path.exists(link):
        os.symlink(os.path.abspath(p), link)
have = {os.path.splitext(f)[0] for f in os.listdir(VIDS)}

items = []
for p in glob.glob(f"{src_root}/**/*.json", recursive=True):
    try:
        data = json.load(open(p, encoding="utf-8"))
    except Exception:
        continue
    if isinstance(data, list):
        items += [it for it in data if isinstance(it, dict) and "match" in str(it.get("task", "")).lower()]
usable = [it for it in items if str(it.get("video_id")) in have]
json.dump(usable, open(QA, "w", encoding="utf-8"))
print(f"{len(have)} videos | {len(items)} AV-matching questions | {len(usable)} with a video on disk")
print("labels:", {lab: sum(str(it["label"]).strip().lower() == lab for it in usable) for lab in ("yes", "no")})
assert usable, "No AV-matching questions with videos found: check the download"

## 4. Check how swapped clips are named (leakage)

Splits are made per *group*, so a clip and its audio-swapped copy must share a group or the test set
leaks training content. Look at the ids below. If swapped clips reuse the original id plus a suffix
(for example `00191` and `00191_swap1`), set `GROUP_REGEX = r"^(.*?)(?:_swap.*)?$"` (adapted to the
real suffix) in the first cell and rerun from here. If the ids are unrelated, leave it empty.

In [ ]:
for lab in ("Yes", "No"):
    ids = [str(it["video_id"]) for it in usable if str(it["label"]).strip().lower() == lab.lower()]
    print(f"{lab} ({len(ids)}):", ids[:15])

## 5. Build the manifest and audit it

In [ ]:
os.makedirs(OUT, exist_ok=True)
MANIFEST = f"{OUT}/manifest.csv"
group = f'--group-regex "{GROUP_REGEX}"' if GROUP_REGEX else ""
!python -m av_contradiction.cli.prepare_dataset avhbench --qa-json {QA} --video-dir {VIDS} --out {MANIFEST} {group}
!python -m av_contradiction.cli.audit_dataset --manifest {MANIFEST} --probe --out {OUT}/audit.json

## 6. Extract CLIP + CLAP features (downloads ~1.2 GB of weights once)

In [ ]:
COMMON = (f"--set experiment_name=avhbench --set data.manifest={MANIFEST} "
          f"--set data.feature_cache_dir={OUT}/cache --set training.checkpoint_dir={OUT}/checkpoints "
          f"--set training.log_dir={OUT}/runs --set evaluation.output_dir={OUT}/reports")
!python -m av_contradiction.cli.extract_features {COMMON} --set encoders.local_files_only=false

## 7. Train every variant and select the best model

Trains the proposed model (`xattn_tensor`), its ablations and the baselines on identical splits and
features, `SEEDS` times each. Every checkpoint is chosen on validation macro-F1; the overall best run
(highest **validation** macro-F1) is copied to `reports/avhbench/best_model/`. Test results never
influence any choice.

In [ ]:
!python -m av_contradiction.cli.run_ablations {COMMON} --seeds {SEEDS}

## 8. Results

In [ ]:
from IPython.display import Image, Markdown, display

rep = f"{OUT}/reports/avhbench"
display(Markdown(open(f"{rep}/ablation_summary.md", encoding="utf-8").read()))
display(Image(f"{rep}/ablation_macro_f1.png"))
print(json.dumps(json.load(open(f"{rep}/best_model/best_model.json")), indent=2))

## 9. Download the best model

Download `best_model.zip` from the notebook's *Output* tab. To test it locally in the dashboard, unzip
it into `output/checkpoints/avhbench_best/` in the project and run
`streamlit run src/av_contradiction/serving/dashboard.py`. CLIP and CLAP must be in the local Hugging
Face cache (they are downloaded once by `extract_features` with `--set encoders.local_files_only=false`).

In [ ]:
print(shutil.make_archive(f"{OUT}/best_model", "zip", f"{OUT}/reports/avhbench/best_model"))